# Fast GECToR と W&I による英語誤り検出・訂正

このノートブックでは、[Fast GECToR](https://github.com/cofe-ai/fast-gector) という文法誤り検出・訂正システムを、英語学習者の作文に誤りの訂正を付けたコーパスで訓練し、英語の誤り訂正をしてみます。

データには、[BEA-2019 共有タスク](https://www.cl.cam.ac.uk/research/nl/bea2019st/)で公開された Write & Improve（W&I）コーパスと LOCNESS コーパス（W&I+LOCNESS v2.1）を使います。W&I は Cambridge English Write & Improve という英作文の学習サイトに投稿された作文に、誤りの訂正を付けたコーパスです。非営利の研究・教育目的なら、申請なしでダウンロードして使えます（ダウンロードした時点で、データに含まれるライセンスに同意したことになります）。

GPU を使うので、[ランタイム] → [ランタイムのタイプを変更] で [ハードウェア アクセラレータ] を `GPU`（T4 など）にしておいてください。

# インストール

Fast GECToR は GECToR の訓練を高速に行う派生版のシステムです。GitHub でソースコードが公開されているので、手元にダウンロードして使いましょう。

In [ ]:
%cd /content
![ -d fast-gector ] || git clone https://github.com/cofe-ai/fast-gector
%cd /content/fast-gector

Fast GECToR には、訓練の最初の数エポックだけエンコーダ（RoBERTa）を固定して出力層だけを訓練する機能（`--cold_step_count`）があります。ただし、固定を解除するときに DeepSpeed の設定ファイルから学習率を読み込む場所が間違っていて、エラーになってしまいます。そこで、ソースコードを1行だけ修正しておきます。

In [ ]:
!sed -i 's/self.lr = config\["optimizer"\].get("lr")/self.lr = config["optimizer"]["params"]["lr"]/' src/trainer.py
!grep -n "self.lr = " src/trainer.py

Fast GECToR は2023年以降更新されておらず、少し古いバージョンの PyTorch（2.0）や DeepSpeed（0.10）、Transformers（4.31）を必要とします。2026年10月時点の Google Colab の Python は 3.13 で、これらのライブラリは Python 3.13 では動かないので、[uv](https://docs.astral.sh/uv/) という高速なパッケージ管理ツールを使って Python 3.10 の仮想環境を `/content/gector-env` に作り、そこにインストールします。数分かかります。最後に `True` と表示されれば GPU が使えます。

なお、以前のバージョンのノートブックでは NVIDIA の apex もインストールしていましたが、Fast GECToR では使われていないので、インストールしないことにしました。

In [ ]:
!pip install -q uv
!uv venv -q --clear -p 3.10 /content/gector-env
# DeepSpeed はインストール時に PyTorch を使うので、先に PyTorch を入れておく
!uv pip install -q -p /content/gector-env torch==2.0.1 numpy==1.25.1 "setuptools<70" wheel packaging ninja
!uv pip install -q -p /content/gector-env --no-build-isolation deepspeed==0.10.0
!uv pip install -q -p /content/gector-env -r requirements.txt
!/content/gector-env/bin/python -c "import sys, torch, deepspeed, transformers; print(sys.version, torch.__version__, deepspeed.__version__, transformers.__version__, torch.cuda.is_available())"

以下では `!deepspeed ...` のようにコマンドを実行するので、作った仮想環境のコマンドが使えるように PATH を設定しておきます。ランタイムを再起動した場合は、このセルをもう一度実行してください。

In [ ]:
%cd /content/fast-gector
import os
if "/content/gector-env/bin" not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = "/content/gector-env/bin:" + os.environ["PATH"]
!which deepspeed

# データの準備

W&I+LOCNESS v2.1 をダウンロードして展開します。ライセンスは `wi+locness/licence.wi.txt` と `wi+locness/license.locness.txt` に書かれています。

In [ ]:
!curl -fsSLO https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz
!tar xzf wi+locness_v2.1.bea19.tar.gz
!ls wi+locness/m2

誤りの訂正は M2 という形式で書かれています。`S` で始まる行が学習者の書いた文（単語分割済み）、`A` で始まる行がその文に対する訂正（何単語目から何単語目までを何に置き換えるか）です。

In [ ]:
!head -n 12 wi+locness/m2/ABC.train.gold.bea19.m2

Fast GECToR の訓練には「誤りを含む文」と「訂正後の文」のペアが必要なので、M2 の訂正を適用して訂正後の文を作ります。

W&I+LOCNESS の訓練データ（`ABC.train`、約3.4万文）を訓練データにします。テストデータの正解は公開されていないので、開発データ（`ABCN.dev`、約4,400文）をシャッフルして半分ずつに分け、開発データとテストデータにします。

In [ ]:
import random

def m2_to_parallel(path):
    """M2 ファイルを（誤りを含む文, 訂正後の文）のペアのリストに変換する"""
    pairs = []
    for block in open(path, encoding="utf-8").read().strip().split("\n\n"):
        lines = block.split("\n")
        src = lines[0][2:].split()  # 先頭の "S " を除く
        trg = list(src)
        offset = 0  # これまでの訂正で増減した単語数
        for line in lines[1:]:
            fields = line[2:].split("|||")  # 先頭の "A " を除く
            start, end = map(int, fields[0].split())
            error_type, correction, annotator = fields[1], fields[2], fields[5]
            if error_type in ("noop", "UNK") or annotator != "0":
                continue
            cor_tokens = correction.split()
            trg[start + offset:end + offset] = cor_tokens
            offset += len(cor_tokens) - (end - start)
        pairs.append((" ".join(src), " ".join(trg)))
    return pairs

train = m2_to_parallel("wi+locness/m2/ABC.train.gold.bea19.m2")
dev_test = m2_to_parallel("wi+locness/m2/ABCN.dev.gold.bea19.m2")
random.seed(0)
random.shuffle(dev_test)
half = len(dev_test) // 2
splits = {"train": train, "dev": dev_test[:half], "test": dev_test[half:]}

for name, pairs in splits.items():
    with open(f"{name}.src", "w", encoding="utf-8") as fs, open(f"{name}.trg", "w", encoding="utf-8") as ft:
        for src, trg in pairs:
            fs.write(src + "\n")
            ft.write(trg + "\n")
    print(name, len(pairs), "文（うち誤りを含む文", sum(src != trg for src, trg in pairs), "）")

In [ ]:
!paste train.src train.trg | head -n 5

GECToR は入力と出力のペアから編集の系列を抽出し、タグとして用います。Fast GECToR にはこの編集系列の抽出のためのスクリプトが含まれているので、こちらを使いましょう（上で作った仮想環境の Python で実行します）。数秒で終わります。

In [ ]:
!python utils/preprocess_data.py -s train.src -t train.trg -o train.edits
!python utils/preprocess_data.py -s dev.src -t dev.trg -o dev.edits
!wc -l train.edits dev.edits test.src test.trg

# モデルの訓練

作成したデータを用いて Fast GECToR モデルの訓練を行います。Fast GECToR は DeepSpeed というライブラリを使って訓練します。DeepSpeed の設定ファイルは `configs/` の下にありますが、Google Colab の GPU（T4）向けに以下のように変更した設定ファイルを用意します。

- `train_batch_size` は256のままで、`gradient_accumulation_steps` を8にして、一度に GPU に載せる文の数を32文にします（GPU のメモリが足りなくならないようにするため）。
- `torch_adam` を `true` にして、PyTorch の Adam を使います（DeepSpeed の Adam を使うと、Google Colab の CUDA のバージョンが合わずにコンパイルに失敗するため）。
- 学習率（`lr`）を元の設定の1e-5から2e-4に上げます。元の設定は、人工的に作った誤りデータで事前に訓練したモデルをさらに訓練するためのものです。この演習のように事前訓練なしで W&I だけで訓練する場合、1e-5 だとほとんど大文字・小文字の訂正しか覚えません。2026年10月に学習率を5e-5から5e-4まで変えて試したところ、2e-4 が一番良く、5e-4 にすると訓練がうまくいきませんでした。

In [ ]:
%%writefile configs/ds_config_colab.json
{
    "train_batch_size": 256,
    "gradient_accumulation_steps": 8,
    "optimizer": {
        "type": "Adam",
        "params": {
            "lr": 2e-4,
            "eps": 1e-8,
            "weight_decay": 1e-2,
            "torch_adam": true
        }
    },
    "gradient_clipping": 1.0,
    "fp16": {
        "enabled": true,
        "auto_cast": false,
        "loss_scale": 0,
        "initial_scale_power": 16,
        "loss_scale_window": 1000,
        "hysteresis": 2,
        "consecutive_hysteresis": false,
        "min_loss_scale": 1
    },
    "zero_optimization": {
        "stage": 1,
        "reduce_bucket_size": 5e8,
        "allgather_bucket_size": 5e8
    }
}

訓練には `train.py` を用います（`scripts/train.sh` の中身とほぼ同じですが、Google Colab 向けにいくつかオプションを変えています）。

- `--num_epochs` は訓練のエポック数です。ここでは10エポックにしています。2026年10月に試したときは、T4 で40分ほどかかりました（最初の2エポックはエンコーダを固定するので速く進みます）。
- `--cold_step_count` は、最初の何エポックの間エンコーダを固定して、新しく追加した出力層だけを `--cold_lr` の学習率で訓練するかを指定します。出力層はランダムな値から始まるので、最初からエンコーダも一緒に訓練するより、出力層をある程度訓練してからエンコーダも訓練するほうが良い結果になります。ここでは GECToR の訓練ガイドと同じく2エポックにしています。
- `--eval_interval` ステップごとに開発データで評価し、`--save_interval` ステップごとにモデルを `ckpts/` の下に保存します。両方を同じ値にしておくと、開発データで一番良かったモデルが必ず保存されます。
- `--tn_prob` は「訂正の必要がない文を訓練に使う確率」をコントロールします。これを変化させることで precision/recall のバランスを取ることができます。いくつかのステージに分けて訓練し、最初の訓練時には `--tn_prob 0` として、最後に適切な値に調整すると良いようです。また、`--skip_correct` もデフォルトは0ですが1にした方が良いようです。詳しくは [GECToR の訓練ガイド](https://github.com/grammarly/gector/blob/master/docs/training_parameters.md)をご覧ください。

訓練の様子は TensorBoard で見ることができます。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
%%bash
set -o pipefail
timestamp=`date "+%Y%m%d_%H%M%S"`
save_dir="ckpts/ckpt_$timestamp"
mkdir -p $save_dir
deepspeed --master_port 49828 train.py \
    --deepspeed \
    --deepspeed_config configs/ds_config_colab.json \
    --num_epochs 10 \
    --max_num_tokens 128 \
    --valid_batch_size 64 \
    --cold_step_count 2 \
    --warmup 0.1 \
    --cold_lr 1e-3 \
    --skip_correct 0 \
    --skip_complex 0 \
    --sub_token_mode average \
    --special_tokens_fix 1 \
    --unk2keep 0 \
    --tp_prob 1 \
    --tn_prob 1 \
    --detect_vocab_path ./data/vocabulary/d_tags.txt \
    --correct_vocab_path ./data/vocabulary/labels.txt \
    --do_eval \
    --train_path train.edits \
    --valid_path dev.edits \
    --save_dir $save_dir \
    --use_cache 0 \
    --log_interval 10 \
    --eval_interval 100 \
    --save_interval 100 \
    --pretrained_transformer_path roberta-base \
    --tensorboard_dir logs/tb/gector_$timestamp \
    2>&1 | tee $save_dir/train.log | grep -v "it/s"

# モデルの評価

訓練したモデルを評価しましょう。訓練されたモデルは `ckpts/ckpt_時刻/globalstep-XXX` のような名前で保存されています。開発データでの学習曲線（TensorBoard）を見るとどのモデルが適しているかがわかります。ここでは、訓練中に記録された評価結果から、開発データでの正解率が一番高かったモデルを選びます。

In [ ]:
import glob, json, os

valid_dirs = [d for d in sorted(glob.glob("ckpts/ckpt_*")) if glob.glob(f"{d}/metrics_globalstep-*.json")]
assert valid_dirs, "metrics_globalstep-*.json が見つかりません"
save_dir = valid_dirs[-1]
metrics_files = sorted(glob.glob(f"{save_dir}/metrics_globalstep-*.json"), key=lambda p: int(p.split("-")[-1].split(".")[0]))
best = json.load(open(metrics_files[-1]))
ckpt_path = f"{save_dir}/globalstep-{best['best_global_step']}"
print(best)
print(ckpt_path, os.path.exists(ckpt_path))
assert os.path.exists(ckpt_path), f"チェックポイントが見つかりません: {ckpt_path}"
os.environ["CKPT_PATH"] = ckpt_path

`predict.py` でテストデータの誤りを訂正します（`scripts/predict.sh` の中身とほぼ同じです）。

`--min_error_probability` は、文に誤りがある確率がこの値より低いときは訂正しないようにする閾値です。値を大きくすると、自信のある訂正だけをするようになるので、precision が上がって recall が下がります。`--additional_confidence` は「訂正しない」というタグの確率に足す値で、これも大きくすると訂正が控えめになります。2026年10月に開発データで試したところ、`--min_error_probability 0.3`、`--additional_confidence 0` が一番良かったので、ここではその値を使います。

In [ ]:
%%bash
deepspeed --include localhost:0 --master_port 42991 predict.py \
    --batch_size 64 \
    --iteration_count 5 \
    --min_seq_len 3 \
    --max_num_tokens 128 \
    --min_error_probability 0.3 \
    --additional_confidence 0.0 \
    --sub_token_mode average \
    --max_pieces_per_token 5 \
    --ckpt_path $CKPT_PATH \
    --deepspeed_config configs/ds_config_colab.json \
    --detect_vocab_path ./data/vocabulary/d_tags.txt \
    --correct_vocab_path ./data/vocabulary/labels.txt \
    --pretrained_transformer_path roberta-base \
    --input_path test.src \
    --out_path test.pred \
    --special_tokens_fix 1 \
    --detokenize 0 \
    --segmented 1 \
    2>&1 | tail -n 5

入力（`test.src`）とシステムの出力（`test.pred`）の差分を見てみましょう。`-` で始まる行が入力、`+` で始まる行がシステムが訂正した文です。

In [ ]:
!diff -u test.src test.pred | head -n 200

## ERRANT による評価

文法誤り訂正では、[ERRANT](https://github.com/chrisjbryant/errant) というツールで、システムの訂正と正解（参照訳）の訂正を比べて precision、recall、F0.5 を計算するのが一般的です。ERRANT をインストールして評価してみましょう。

2026年10月に T4 で試したときは、precision が0.51、recall が0.24、F0.5 が0.41でした（実行のたびに結果は少し変わり、A100 で試したときは F0.5 が0.44でした）。最初のバージョンのノートブック（学習率5e-5、5エポック、`--cold_step_count 0`）では F0.5 が0.26～0.28程度だったので、学習率・エポック数・`--cold_step_count`・`--min_error_probability` を調整することで大きく改善しています。さらに、cLang-8 のような大きなデータで先に訓練してから W&I で訓練すると、もっと良くなります。

In [ ]:
# ERRANT は Google Colab の Python にインストールする（PATH の先頭にある仮想環境の Python ではなく）
import sys
!{sys.executable} -m pip install -q errant
!{sys.executable} -m spacy download en_core_web_sm
!errant_parallel -orig test.src -cor test.pred -out hyp.m2
!errant_parallel -orig test.src -cor test.trg -out ref.m2
!errant_compare -hyp hyp.m2 -ref ref.m2